# Exercise 2b: Feature engineering

In [71]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import re
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder
from sklearn.impute import SimpleImputer

In [72]:
X_train = pd.read_csv("ex2_train.csv")
y_train = pd.read_csv("ex2_class_train.csv")
X_test = pd.read_csv("ex2_test.csv")
y_test = pd.read_csv("ex2_class_test.csv")

In [73]:
# define a utility function to print out the prediction performance
def evaluate_result(y_test, y_pred, clf):
    print(f'Accuracy: {accuracy_score(y_test, y_pred):.4f}')
    print(f'Precision: {precision_score(y_test, y_pred):.4f}')
    print(f'Recall: {recall_score(y_test, y_pred):.4f}')
    print(f'F1-score: {f1_score(y_test, y_pred):.4f}')
    print(f'AUC-ROC: {roc_auc_score(y_test, clf.predict_proba(X_test_processed)[:, 1]):.4f}')

In [74]:
def compute_metrics(y_true, y_pred, clf, X_test):
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred),
        "Recall": recall_score(y_true, y_pred),
        "F1-score": f1_score(y_true, y_pred),
        "AUC-ROC": roc_auc_score(y_true, clf.predict_proba(X_test)[:, 1]),
    }

## Prototyping (without feature engineering)

In [75]:
def preprocess(data_in):
    data = data_in.drop(columns=['Name'])
    
    data = data.fillna({
        'Age': data['Age'].median(),
        'Embarked': data['Embarked'].mode(dropna=True).iloc[0],
        'Fare': data['Fare'].median()
    })

    # Convert categorical variables to dummy/indicator variables
    data = pd.get_dummies(data, columns=['Sex', 'Embarked'], drop_first=True)

    return data

In [76]:
X_train_processed = preprocess(X_train)
X_test_processed = preprocess(X_test)

clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train_processed, y_train.values.ravel())
y_pred = clf.predict(X_test_processed)

print('Random Forest Model without Feature Engineering')
evaluate_result(y_test, y_pred, clf)

baseline = compute_metrics(y_test, y_pred, clf, X_test_processed)

Random Forest Model without Feature Engineering
Accuracy: 0.8101
Precision: 0.7778
Recall: 0.7568
F1-score: 0.7671
AUC-ROC: 0.8732


## Feature engineering

The classification using simple preprocessed data gives only mediocre performance.

**TODO: You should make use of the insights from your EDA (ex2a) to complete the following feature engineering function below.** Later the function will replace the simple preprocessing.

You will pass the exercise if your feature engineering can improve the performance (i.e., winning in three or more metrics).

In [77]:
def feature_engineering(data_in):
    df = data_in.copy()
    df = df.drop(columns=["Name"])

    df["Age"] = df["Age"].fillna(df["Age"].median())
    df["Fare"] = df["Fare"].fillna(df["Fare"].median())
    df["Embarked"] = df["Embarked"].fillna(df["Embarked"].mode().iloc[0])

    df["FamilySize"] = df["SibSp"] + df["Parch"] + 1
    df["IsAlone"] = (df["FamilySize"] == 1).astype(int)
    df["SmallFamily"] = df["FamilySize"].between(2, 4).astype(int) 
    df["IsChild"] = (df["Age"] < 16).astype(int)                   

    df["Sex"] = df["Sex"].map({"male": 0, "female": 1})
    df = pd.get_dummies(df, columns=["Embarked"], drop_first=True)

    return df

In [78]:
X_train_processed = feature_engineering(X_train)
X_test_processed = feature_engineering(X_test)

clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train_processed, y_train.values.ravel())
y_pred = clf.predict(X_test_processed)

print('Random Forest Model with Feature Engineering')
evaluate_result(y_test, y_pred, clf)

Random Forest Model with Feature Engineering
Accuracy: 0.8156
Precision: 0.7887
Recall: 0.7568
F1-score: 0.7724
AUC-ROC: 0.8792


In [79]:
engineered = compute_metrics(y_test, y_pred, clf, X_test_processed)

cmp = pd.DataFrame({"baseline": baseline, "engineered": engineered})
cmp["better"] = cmp["engineered"] > cmp["baseline"]
print(cmp.round(4))
print(f"Wins: {cmp['better'].sum()} / {len(cmp)}")

           baseline  engineered  better
Accuracy     0.8101      0.8156    True
Precision    0.7778      0.7887    True
Recall       0.7568      0.7568   False
F1-score     0.7671      0.7724    True
AUC-ROC      0.8732      0.8792    True
Wins: 4 / 5
